In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import pathlib
import numpy as np

In [2]:
df = pd.read_csv("../database/players.csv")

df.head()

,name,role,team,hltv,major_wins,win,second,semi,quarter,no_major_teammates,no_teammates,no_events
0,donk,Opener,Spirit,1.40,0,0,0,0,0,0,0,0
1,magixx,Opener,Spirit,0.97,0,3,1,2,3,0,4,9
2,sh1ro,AWPer,Spirit,1.17,0,0,0,0,0,0,0,0
3,tN1R,Closer,Spirit,1.05,0,0,0,0,0,0,0,0
4,zont1x,Support,Spirit,0.99,0,0,0,0,0,0,0,0


In [3]:
filename = "gam.pkl"

with pathlib.Path(filename).open("rb") as f:
    gam = pickle.load(f)

In [4]:
df["events_bonus"] = df["no_events"]**(0.1) / 15
df["perf_bonus"] = (8*df["win"] + 4*df["second"] + 2*df["semi"] + df["quarter"])/(12*df["no_events"]**0.5)
df["team_bonus"] = 0.5*np.sqrt(df["no_teammates"])
df["p"] = gam.predict_proba(df[["hltv"]])

df["igl_bonus"] = df["events_bonus"] + (df["perf_bonus"] + df["team_bonus"]) / (15*df["p"]**0.5)

df["total"] = 0.8*df["hltv"] + df["igl_bonus"]

igl = df[df["no_events"] > 0]
cols = ["name", "team", "hltv", "p", "events_bonus", "perf_bonus", "team_bonus", "igl_bonus", "total"]
igl = igl[cols]

igl.sort_values("total", ascending=False)

,name,team,hltv,p,events_bonus,perf_bonus,team_bonus,igl_bonus,total
5,karrigan,Falcons,0.77,0.248015,0.108385,2.127756,2.121320,0.677191,1.293191
15,apEX,Vitality,0.97,0.436717,0.100889,2.540761,1.500000,0.508524,1.284524
10,FalleN,FURIA,0.93,0.401351,0.106671,1.755960,1.581139,0.457840,1.201840
35,Aleksib,Natus Vincere,0.88,0.353537,0.100889,1.375371,1.500000,0.423282,1.127282
80,Jame,PARIVISION,1.05,0.498565,0.095660,0.602796,1.414214,0.286099,1.126099
77,Boombl4,BetBoom,0.94,0.410457,0.090814,1.083770,1.118034,0.319929,1.071929
1,magixx,Spirit,0.97,0.436717,0.083049,0.972222,1.000000,0.282008,1.058008
142,Brollan,HEROIC,0.97,0.436717,0.089952,0.857159,1.000000,0.277304,1.053304
48,kyxsan,Aurora,0.95,0.419388,0.094854,0.786035,1.000000,0.278716,1.038716
65,bLitz,MongolZ,0.95,0.419388,0.095130,0.577522,1.000000,0.257526,1.017526


In [5]:
df["events_bonus"] = df["no_events"]**(0.1) / 15
df["perf_bonus"] = (8*df["win"] + 4*df["second"] + 2*df["semi"] + df["quarter"])/(12*df["no_events"]**0.5)
df["team_bonus"] = 0.5*np.sqrt(df["no_teammates"])
df["p"] = gam.predict_proba(df[["hltv"]])

df["igl_bonus"] = df["events_bonus"] + (df["perf_bonus"] + df["team_bonus"]) / (15*df["p"]**0.5)
df = df.fillna(0)

df["igl_score"] = 0.8*df["hltv"] + df["igl_bonus"]

df.sort_values("igl_score", ascending=False)

,name,role,team,hltv,major_wins,win,second,semi,quarter,no_major_teammates,no_teammates,no_events,events_bonus,perf_bonus,team_bonus,p,igl_bonus,total,igl_score
5,karrigan,Opener,Falcons,0.77,2,16,17,24,46,8,18,129,0.108385,2.127756,2.121320,0.248015,0.677191,1.293191,1.293191
15,apEX,Opener,Vitality,0.97,3,22,7,12,14,7,9,63,0.100889,2.540761,1.500000,0.436717,0.508524,1.284524,1.284524
10,FalleN,Opener,FURIA,0.93,2,12,10,26,33,4,10,110,0.106671,1.755960,1.581139,0.401351,0.457840,1.201840,1.201840
35,Aleksib,Closer,Natus Vincere,0.88,1,7,10,8,19,4,9,63,0.100889,1.375371,1.500000,0.353537,0.423282,1.127282,1.127282
80,Jame,AWPer,PARIVISION,1.05,1,2,2,4,12,4,8,37,0.095660,0.602796,1.414214,0.498565,0.286099,1.126099,1.126099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,zeRRoFIX,Closer,Inner Circle,0.75,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.230602,0.000000,0.000000,0.600000
159,rain,Support,100 Thieves,0.71,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.198250,0.000000,0.000000,0.568000
134,0SAMAS,Support,JiJieHao,0.70,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.190704,0.000000,0.000000,0.560000
156,poiii,Closer,100 Thieves,0.70,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.190704,0.000000,0.000000,0.560000


In [13]:
count = 165
df["odds"] = np.log(df["p"] / (1-df["p"]))
df["p_igl"] = gam.predict_proba(df[["igl_score"]])
df["igl_odds"] = np.log(df["p_igl"] / (1-df["p_igl"]))

avg_score = df[df["no_events"] == 0]["odds"].mean()
avg_igl = df[df["no_events"] > 0]["igl_odds"].mean()

print(avg_score)
print(avg_igl)

-0.20703908861700385
-0.5389262968894004


In [14]:
team_avg_score = 4*avg_score + avg_igl
print(team_avg_score)

-1.367082651357416


In [24]:
team = df[df["name"].isin({"mezii", "ZywOo", "flameZ", "ropz", "apEX"})]
team_b = team[team["no_events"] == 0]["odds"].sum() + team[team["name"] == "apEX"]["igl_odds"].sum()
print(team_b)

1.5613020936020154


In [26]:
team = df[df["name"].isin({"TeSeS", "m0NESY", "NiKo", "karrigan", "kyousuke"})]
print(team["name"])
team_b = team[team["name"] != "karrigan"]["odds"].sum() + team[team["name"] == "karrigan"]["igl_odds"].sum()
print(team_b)

5    karrigan
6        NiKo
7       TeSeS
8      m0NESY
9    kyousuke
Name: name, dtype: str
1.3356948153465982


In [28]:
team = df[df["name"].isin({"yuurih", "molodoy", "KSCERATO", "FalleN", "YEKINDAR"})]
print(team["name"])
team_b = team[team["name"] != "FalleN"]["odds"].sum() + team[team["name"] == "FalleN"]["igl_odds"].sum()
print(team_b)

10      FalleN
11      yuurih
12    YEKINDAR
13    KSCERATO
14     molodoy
Name: name, dtype: str
0.6488043168298789


In [30]:
names = df[df["team"] == "MOUZ"]["name"]
print(names)
team = df[df["name"].isin(names)]
team_b = team[team["name"] != "xertioN"]["odds"].sum() + team[team["name"] == "xertioN"]["igl_odds"].sum()
print(team_b)

20     torzsi
21      Spinx
22    xertioN
23         PR
24      xelex
Name: name, dtype: str
-0.44335290009785816


In [34]:
names = df[df["team"] == "FUT"][["name", "no_events"]]
print(names)
team = df[df["name"].isin(names["name"])]
team_b = team[team["name"] != "Krabeni"]["odds"].sum() + team[team["name"] == "Krabeni"]["igl_odds"].sum()
print(team_b)

       name  no_events
25   xfl0ud          0
26    dem0n          0
27  Krabeni          8
28    cmtry          0
29  dziugss          0
-0.7385929079323773


In [37]:
names = df[df["team"] == "B8"][["name", "no_events"]]
print(names)
team = df[df["name"].isin(names["name"])]
team_b = team[team["name"] != "alex666"]["odds"].sum() + team[team["name"] == "alex666"]["igl_odds"].sum()
print(team_b)

         name  no_events
70    alex666          7
71        npl          0
72   kensizor          0
73  esenthial          0
74      s1zzi          0
-1.7591990561825335


In [38]:
names = df[df["team"] == "100 Thieves"][["name", "no_events"]]
print(names)
team = df[df["name"].isin(names["name"])]
team_b = team[team["name"] != "Gizmy"]["odds"].sum() + team[team["name"] == "Gizmy"]["igl_odds"].sum()
print(team_b)

       name  no_events
155  device          0
156   poiii          0
157   Gizmy          4
158   sirah          0
159    rain          0
-6.115837924873549


In [45]:
scores = []
teams = df["team"].unique()
for team in teams:
    players = df[df["team"] == team][["name", "odds", "igl_odds", "no_events"]].sort_values(by="no_events", ascending=False)
    igl = players.iloc[0]
    score = players[players["name"] != igl["name"]]["odds"].sum() + igl["igl_odds"] - team_avg_score
    scores.append((team, score))
print(scores)

[('Spirit', np.float64(2.2489263805017927)), ('Falcons', np.float64(2.7027774667040143)), ('FURIA', np.float64(2.015886968187295)), ('Vitality', np.float64(2.9283847449594314)), ('MOUZ', np.float64(0.9237297512595577)), ('FUT', np.float64(0.6284897434250386)), ('Legacy', np.float64(0.5235583663426044)), ('Natus Vincere', np.float64(2.079897165990542)), ('G2', np.float64(1.256530397836538)), ('Aurora', np.float64(1.3493717323097658)), ('9z', np.float64(-0.014099796398995057)), ('FaZe', np.float64(0.4877646950846388)), ('Astralis', np.float64(1.1396880945400019)), ('MongolZ', np.float64(-0.0886267293641565)), ('B8', np.float64(-0.39211640482511756)), ('BetBoom', np.float64(0.6005718606716122)), ('PARIVISION', np.float64(0.4866428825820981)), ('GamerLegion', np.float64(1.2575037475561346)), ('paiN', np.float64(-1.0780409078355433)), ('magic', np.float64(0.13586579957763334)), ('3DMAX', np.float64(-2.0884025495028897)), ('Liquid', np.float64(-0.5148482512662218)), ('BIG', np.float64(-1.446

In [50]:
scores.sort(key=lambda x: x[1], reverse=True)
print(scores)

[('Vitality', np.float64(2.9283847449594314)), ('Falcons', np.float64(2.7027774667040143)), ('Spirit', np.float64(2.2489263805017927)), ('Natus Vincere', np.float64(2.079897165990542)), ('FURIA', np.float64(2.015886968187295)), ('Aurora', np.float64(1.3493717323097658)), ('GamerLegion', np.float64(1.2575037475561346)), ('G2', np.float64(1.256530397836538)), ('Astralis', np.float64(1.1396880945400019)), ('MOUZ', np.float64(0.9237297512595577)), ('FUT', np.float64(0.6284897434250386)), ('BetBoom', np.float64(0.6005718606716122)), ('Legacy', np.float64(0.5235583663426044)), ('FaZe', np.float64(0.4877646950846388)), ('PARIVISION', np.float64(0.4866428825820981)), ('Ninjas in Pyjamas', np.float64(0.37186438557545043)), ('magic', np.float64(0.13586579957763334)), ('9z', np.float64(-0.014099796398995057)), ('MongolZ', np.float64(-0.0886267293641565)), ('HEROIC', np.float64(-0.1754104963228622)), ('MIBR', np.float64(-0.18255110590805845)), ('B8', np.float64(-0.39211640482511756)), ('Liquid', n

In [52]:
best_team = df[df["name"].isin({"karrigan", "ZywOo", "donk", "KSCERATO", "HeavyGod"})][["name", "odds", "igl_odds"]]
best_score = best_team[best_team["name"] != "karrigan"]["odds"].sum() + best_team[best_team["name"] == "karrigan"]["igl_odds"] - team_avg_score

print(best_score)

5    3.9217
Name: igl_odds, dtype: float64


In [53]:
factor = 10/best_score
print(factor)

5    2.549915
Name: igl_odds, dtype: float64
